# Notebook 2 — Clustering (R2)
Rice Variety Classification | F20DL 2026/2027

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

df = pd.read_csv('../data/tabular/rice_cammeo_osmancik.csv')
X = df.drop(columns=['Class'])
y = df['Class']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 1. K-Means Clustering — Elbow Method

In [ ]:
inertias, sil_scores = [], []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(2, 10), inertias, marker='o')
ax1.set(xlabel='k', ylabel='Inertia', title='Elbow Method')
ax2.plot(range(2, 10), sil_scores, marker='o', color='orange')
ax2.set(xlabel='k', ylabel='Silhouette Score', title='Silhouette Scores')
plt.tight_layout()
plt.savefig('../results/kmeans_elbow.png', dpi=100)
plt.show()

## 2. K-Means with k=2

In [ ]:
km = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km = km.fit_predict(X_scaled)
print('Silhouette Score (KMeans):', silhouette_score(X_scaled, labels_km))
print('ARI (KMeans):', adjusted_rand_score(y, labels_km))

## 3. EM / Gaussian Mixture Model

In [ ]:
gmm = GaussianMixture(n_components=2, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)
print('Silhouette Score (GMM):', silhouette_score(X_scaled, labels_gmm))
print('ARI (GMM):', adjusted_rand_score(y, labels_gmm))

## 4. PCA Visualisation of Clusters

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}')

le = LabelEncoder()
y_enc = le.fit_transform(y)

km_final = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km = km_final.fit_predict(X_scaled)

gmm = GaussianMixture(n_components=2, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, labels, title in zip(axes,
                              [y_enc, labels_km, labels_gmm],
                              ['Ground Truth', 'K-Means (k=2)', 'GMM (EM)']):
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='Set1', alpha=0.4, s=10)
    ax.set(title=title, xlabel='PC1', ylabel='PC2')
    ax.legend(handles=scatter.legend_elements()[0], labels=['Class 0', 'Class 1'])

plt.tight_layout()
plt.savefig('../results/clustering_pca.png', dpi=100)
plt.show()

print(f'\nKMeans  — Silhouette: {silhouette_score(X_scaled, labels_km):.4f}  ARI: {adjusted_rand_score(y, labels_km):.4f}')
print(f'GMM/EM  — Silhouette: {silhouette_score(X_scaled, labels_gmm):.4f}  ARI: {adjusted_rand_score(y, labels_gmm):.4f}')